# Compressor thermal network and condensate-evaporation screening

This example selects a generic compressor from the NeqSim compressor catalog, calculates local screening temperatures, and checks whether an entrained hydrocarbon-condensate droplet would partly evaporate at the inlet-shaft temperature. Such evaporation can concentrate dissolved elemental sulfur at the metal surface.

> The built-in thermal conductances and heat capacities are illustrative. Replace them with OEM geometry, vendor loss data, oil/seal measurements, or fitted plant data before making integrity or operating decisions.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'devtools' / 'neqsim_dev_setup.py').exists():
    root = root.parent
sys.path.insert(0, str(root / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(recompile=True))

CompressorCatalog = ns.JClass('neqsim.process.equipment.compressor.CompressorCatalog')
CompressorThermalModel = ns.JClass('neqsim.process.equipment.compressor.CompressorThermalModel')
ThermodynamicOperations = ns.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations')

## Gas and compressor

The process gas is deliberately rich enough to represent a recompressor service where a small liquid carry-over is credible. The process simulation supplies suction and discharge temperatures to the thermal network.

In [ ]:
gas = ns.SystemSrkEos(273.15 + 25.0, 50.0)
for component, amount in [('methane', 0.88), ('ethane', 0.06), ('propane', 0.03),
                          ('n-butane', 0.015), ('n-pentane', 0.010), ('n-hexane', 0.005)]:
    gas.addComponent(component, amount)
gas.setMixingRule('classic')
inlet = ns.Stream('recompressor inlet', gas)
inlet.setFlowRate(1.0, 'MSm3/day')
inlet.run()

compressor = ns.Compressor('recompressor', inlet)
compressor.setOutletPressure(100.0, 'bara')
compressor.setIsentropicEfficiency(0.78)
losses = compressor.initMechanicalLosses(120.0)
losses.setSealGasSupplyTemperature(40.0)
losses.setLubeOilInletTemp(40.0)
losses.setLubeOilOutletTemp(55.0)

catalog = CompressorCatalog.createDefaultCatalog()
compressor.applyCatalogEntry(catalog, 'generic-centrifugal-single-stage')
compressor.run()

In [ ]:
model = compressor.getThermalModel()
profile_c = {str(e.getKey()): float(e.getValue()) for e in model.getTemperatureProfile('C').entrySet()}
for location, temperature in profile_c.items():
    print(f'{location:20s} {temperature:8.2f} degC')

## Entrained-condensate evaporation at the inlet shaft

A separate representative condensate is flashed at suction pressure and at the calculated shaft temperature. The resulting equilibrium gas fraction is a screening indicator for local evaporation. It is not a droplet heat/mass-transfer model, and elemental-sulfur solubility must be evaluated separately.

In [ ]:
condensate = ns.SystemSrkEos(273.15 + 25.0, inlet.getPressure('bara'))
for component, amount in [('n-butane', 0.20), ('n-pentane', 0.30),
                          ('n-hexane', 0.30), ('n-heptane', 0.20)]:
    condensate.addComponent(component, amount)
condensate.setMixingRule('classic')
ops = ThermodynamicOperations(condensate)

def equilibrium_gas_fraction(temperature_c):
    condensate.setTemperature(temperature_c, 'C')
    condensate.setPressure(inlet.getPressure('bara'), 'bara')
    ops.TPflash()
    return float(condensate.getPhaseFraction('gas', 'mole')) if condensate.hasPhaseType('gas') else 0.0

shaft_c = profile_c['inlet-shaft']
gas_fraction_inlet = equilibrium_gas_fraction(inlet.getTemperature('C'))
gas_fraction_shaft = equilibrium_gas_fraction(shaft_c)
print(f'Condensate equilibrium gas fraction at inlet: {gas_fraction_inlet:.4f}')
print(f'Condensate equilibrium gas fraction at shaft: {gas_fraction_shaft:.4f}')
print(f'Additional equilibrium evaporation potential: {gas_fraction_shaft-gas_fraction_inlet:.4f}')

## Machine-specific calibration inputs

The catalog keeps the inputs that should be replaced for a real machine next to the model, so a UI or engineering workflow can expose them directly.

In [ ]:
entry = catalog.get('generic-centrifugal-single-stage')
for e in entry.getRequiredParameters().entrySet():
    print(f'- {e.getKey()}: {e.getValue()}')

# Example calibration: change one effective conductance, then resolve.
for link in model.getLinks():
    connected = {str(link.getFromNodeId()), str(link.getToNodeId())}
    if connected == {'inlet-shaft', 'impeller'}:
        link.setConductanceWPerK(500.0)
compressor.solveThermalModel()
print('Calibrated inlet-shaft temperature:',
      model.getTemperature(CompressorThermalModel.INLET_SHAFT, 'C'), 'degC')

## Transient temperature

For start-up, shutdown, hot restart, or changing seal/oil conditions, disable automatic steady-state solving and advance the implicit transient model with the process time step.

In [ ]:
model.setAutoSolve(False)
shaft_history = []
for minute in range(61):
    shaft_history.append((minute, model.getTemperature('inlet-shaft', 'C')))
    compressor.solveThermalModelTransient(60.0)

try:
    import matplotlib.pyplot as plt
    plt.plot([x[0] for x in shaft_history], [x[1] for x in shaft_history])
    plt.xlabel('Time [min]')
    plt.ylabel('Inlet-shaft temperature [degC]')
    plt.grid(True)
except ImportError:
    print(shaft_history[:5], '...', shaft_history[-1])